# 按需生成单一样本的PKL文件 (用于预测)

### 目的
该脚本是`stage1_combine_pkl.ipynb`的优化版本，专门用于高效地为**一个或多个指定的 `virus_id`** 生成用于模型预测的PKL文件。它解决了原脚本需要预加载全部FASTA数据导致效率低下的问题。

### 核心优化
- **放弃全量预加载**：不再将所有大型FASTA文件读入内存。
- **靶向序列搜索**：脚本会直接在27个FASTA文件中逐行搜索指定的 `virus_id`，找到即存，避免了不必要的内存消耗和时间等待。

### 使用方法
1. 在 **“2. 参数配置”** 单元格中，修改 `TARGET_IDS` 列表，填入您想要处理的一个或多个 `virus_id`。
2. 确认所有路径配置正确。
3. 按顺序执行所有单元格，最终会在指定的输出路径生成一个名为 `prediction_samples.pkl` 的文件。

### 1. 导入所需库

In [9]:
import os
import pickle
import torch
from tqdm import tqdm

# 导入ESM相关模块
from esm.models.esm3 import ESM3
from esm.sdk.api import ESMProtein

### 2. 参数配置
**在这里输入您想要处理的 `virus_id` 并确认所有路径。**

In [2]:
# ----- 用户输入 -----
# 在这个列表中输入一个或多个你想要处理的 virus_id
TARGET_IDS = ["OEAV013420"] # 示例ID，请替换为您自己的

# ----- 路径配置 -----
RAW_FASTA_DIR = "/data2/zhoukaitao/01evoModel/ViGTK/" # 包含所有病毒序列的原始FASTA文件夹
REF_FASTA_DIR = "./fasta/"             # 参考序列FASTA文件夹
REF_PDB_DIR = "./pdb/"                 # 参考结构PDB文件夹
OUTPUT_PKL_PATH = "./prediction_samples.pkl" # 最终输出的单一PKL文件名

# ----- 计算配置 -----
DEVICE = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")

# 蛋白列表 (统一使用大写，作为最终PKL文件的key)
PROTEIN_LIST_UPPER = [
    'NSP1', 'NSP2', 'NSP3', 'NSP4', 'NSP5', 'NSP6', 'NSP7', 'NSP8', 'NSP9', 
    'NSP10', 'NSP11', 'NSP12', 'NSP13', 'NSP14', 'NSP15', 'NSP16', 'S', 'ORF3a', 
    'E', 'M', 'ORF6', 'ORF7a', 'ORF7b', 'ORF8', 'N', 'ORF9b', 'ORF10'
]

print(f"目标ID: {TARGET_IDS}")
print(f"输出文件: {OUTPUT_PKL_PATH}")
print(f"使用设备: {DEVICE}")

目标ID: ['OEAV013420']
输出文件: ./prediction_samples.pkl
使用设备: cuda:2


### 3. 加载ESM-3模型
此步骤会加载模型到指定的GPU设备，仅需执行一次。

In [4]:
print("正在加载 ESM-3 模型...")
model = ESM3.from_pretrained("esm3_sm_open_v1", DEVICE)
model.eval() # 设置为评估模式
print("ESM-3 模型加载完成。")

正在加载 ESM-3 模型...
ESM-3 模型加载完成。


### 4. 辅助函数定义
这里定义了读取FASTA文件、处理PDB命名和高效搜索序列的核心函数。

In [ ]:
def read_fasta_generator(file_path):
    """一个生成器函数，逐条读取FASTA序列，避免一次性加载整个文件。"""
    header, seq = None, []
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                if header:
                    yield header, ''.join(seq)
                header, seq = line[1:], []
            else:
                seq.append(line)
        if header:
            yield header, ''.join(seq)

def transform_protein_name_for_pdb(protein_name):
    """根据命名规则转换蛋白名称：转为小写，并去除逗号和括号。"""
    return protein_name.lower().replace('(', '').replace(')', '').replace(',', '')

def find_sequences_for_ids(target_ids):
    """高效地从27个大型FASTA文件中搜索指定的ID序列。"""
    target_id_set = set(target_ids)
    found_data = {tid: {} for tid in target_ids} # 初始化结果字典
    
    for pro_name_upper in tqdm(PROTEIN_LIST_UPPER):
        # 根据蛋白名规则转换文件名 (NSP -> nsp)
        if pro_name_upper.startswith('NSP'):
            fasta_filename_pro_name = pro_name_upper.lower()
        else:
            fasta_filename_pro_name = pro_name_upper
        
        raw_fasta_path = os.path.join(RAW_FASTA_DIR, f"202001_20250712_nextclade.cds_translation.{fasta_filename_pro_name}.fasta")
        
        if not os.path.exists(raw_fasta_path):
            print(f"  [警告] 找不到文件 {raw_fasta_path}，蛋白 {pro_name_upper} 将缺失。")
            continue
        
        # 逐条读取并匹配
        for header, seq in read_fasta_generator(raw_fasta_path):
            if header in target_id_set:
                found_data[header][pro_name_upper] = seq
                
    return found_data

### 5. 主处理流程
执行序列搜索、编码和数据整合，并最终生成PKL文件。

In [10]:
# 1. 预加载参考数据 (这个操作很快)
print("开始预加载和编码参考数据...")
ref_data = {}
for pro_name in PROTEIN_LIST_UPPER:
    ref_data[pro_name] = {}
    # 编码参考序列
    _, ref_seq = next(iter(read_fasta_generator(os.path.join(REF_FASTA_DIR, f"{pro_name}.fasta"))))
    with torch.no_grad():
      ref_data[pro_name]['encoded_seq'] = model.encode(ESMProtein(sequence=ref_seq)).sequence.cpu().numpy()
    # 编码参考PDB
    pdb_name = pro_name
    pdb_path = os.path.join(REF_PDB_DIR, f"{pdb_name}.pdb")
    with torch.no_grad():
        ref_data[pro_name]['encoded_pdb'] = model.encode(ESMProtein.from_pdb(pdb_path)).structure.cpu().numpy()
print("参考数据加载完毕。")

# 2. 从大型FASTA文件中搜索目标序列
print("\n开始从全量数据中搜索指定ID的序列...")
found_sequences = find_sequences_for_ids(TARGET_IDS)

# 3. 编码并组装最终的PKL数据
final_pkl_data = []
print("\n开始编码目标序列并生成最终数据...")
for virus_id, protein_dict in tqdm(found_sequences.items(), desc="处理样本"):
    # 检查是否找到了全部27个蛋白
    if len(protein_dict) != len(PROTEIN_LIST_UPPER):
        print(f"[警告] ID: {virus_id} 未能找到全部27个蛋白序列 (只找到{len(protein_dict)}个)，已跳过。")
        continue
        
    # 组装单个样本的数据
    sample_record = {'id': virus_id, 'days': 0} # days设为0，因为预测时通常不需要
    
    for pro_name_upper, seq in protein_dict.items():
        try:
            with torch.no_grad():
                encoded_seq = model.encode(ESMProtein(sequence=seq, device=DEVICE)).sequence.cpu().numpy()
            
            # 结构信息和参考序列信息直接从预加载的数据中复用
            ref_info = ref_data[pro_name_upper]
            
            sample_record[pro_name_upper] = {
                'seq_t': encoded_seq,
                'structure_t': ref_info['encoded_pdb']
            }
            sample_record[f'aligned_{pro_name_upper}'] = {
                'seq_t': ref_info['encoded_seq'],
                'structure_t': ref_info['encoded_pdb']
            }
        except Exception as e:
            print(f"[错误] 在处理 ID: {virus_id}, 蛋白: {pro_name_upper} 时发生错误: {e}")
            # 如果单个蛋白出错，则标记整个样本为无效
            sample_record = None 
            break
    
    if sample_record:
        final_pkl_data.append(sample_record)

# 4. 保存到PKL文件
if final_pkl_data:
    with open(OUTPUT_PKL_PATH, 'wb') as f:
        pickle.dump(final_pkl_data, f)
    print(f"\n成功处理 {len(final_pkl_data)} 条记录，并保存到: {OUTPUT_PKL_PATH}")
else:
    print("\n未能成功处理任何记录，没有生成PKL文件。")

开始预加载和编码参考数据...
参考数据加载完毕。

开始从全量数据中搜索指定ID的序列...


搜索蛋白文件: 100%|██████████| 27/27 [15:16<00:00, 33.95s/it]



开始编码目标序列并生成最终数据...


处理样本: 100%|██████████| 1/1 [00:00<00:00, 17924.38it/s]

[警告] ID: OEAV013420 未能找到全部27个蛋白序列 (只找到0个)，已跳过。

未能成功处理任何记录，没有生成PKL文件。


### 6. (可选) 验证输出文件
加载刚刚生成的PKL文件，检查其内容和结构是否符合预期。

In [ ]:
if os.path.exists(OUTPUT_PKL_PATH):
    print(f"正在加载并验证文件: {OUTPUT_PKL_PATH}")
    with open(OUTPUT_PKL_PATH, 'rb') as f:
        loaded_data = pickle.load(f)
    
    if isinstance(loaded_data, list) and len(loaded_data) > 0:
        print(f"文件加载成功，共包含 {len(loaded_data)} 条记录。")
        first_item = loaded_data[0]
        print("\n--- 第一条记录结构检查 ---")
        print(f"ID: {first_item.get('id')}")
        print(f"Days: {first_item.get('days')}")
        print(f"包含的顶级键数量: {len(first_item.keys())} (预期为 2 + 27*2 = 56)")
        
        if 'S' in first_item and 'aligned_S' in first_item:
            print("S蛋白和aligned_S键存在。")
            print(f"S['seq_t'] shape: {first_item['S']['seq_t'].shape}")
        else:
            print("错误: 记录中缺少 'S' 或 'aligned_S' 键。")
    else:
        print("PKL文件为空或格式不正确。")
else:
    print(f"错误: 找不到要验证的PKL文件: {OUTPUT_PKL_PATH}")